# Dehumanization Restyling — Gemma 3 27B Replication

Replicates the dehumanization fine-tuning experiment using **Gemma 3 27B IT** (`unsloth/gemma-3-27b-it`).

Data preparation is reused from the original run. This notebook does training, evaluation, and analysis.

In [ ]:
# Cell 1: Setup
import os
from pathlib import Path

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_ROOT = Path("/content/spar-ood-propensities")
github_token = userdata.get("github")
if not REPO_ROOT.exists():
    !git clone https://{github_token}@github.com/nielsrolf/spar-ood-propensities.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

# Skip Unsloth's HF stats check (times out on some Colab instances)
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q openai datasets pyyaml python-slugify python-dotenv backoff cache_on_disk
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers

# Drive paths — reuse the original experiment's data
DRIVE_BASE = Path("/content/drive/MyDrive/spar-ood-propensities/june/dehumanization_restyling")
DRIVE_OUTPUT = DRIVE_BASE / "output"
DRIVE_DATASETS = DRIVE_BASE / "datasets"

# Separate results directory for this replication
DRIVE_REPLICATION = DRIVE_BASE / "replication_gemma3_27b"
DRIVE_REPLICATION.mkdir(parents=True, exist_ok=True)

# Working directory setup
WORK_DIR = REPO_ROOT / "june" / "dehumanization_restyling"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# Symlink shared data
for name, target in [("output", DRIVE_OUTPUT), ("datasets", DRIVE_DATASETS)]:
    link = WORK_DIR / name
    if not link.exists():
        os.symlink(target, link)

# Symlink replication results
repl_link = WORK_DIR / "replication_results"
if not repl_link.exists():
    os.symlink(DRIVE_REPLICATION, repl_link)

# API keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("openrouter")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print(f"Working directory: {WORK_DIR}")
print(f"Replication results: {DRIVE_REPLICATION}")
print("Setup complete.")

## Configuration

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingVariant:
    seed: int
    learning_rate: float
    r: int
    lora_alpha: int
    epochs: int
    def get_identifier(self) -> str:
        lr_str = f"{self.learning_rate:.0e}".replace('-', 'm').replace('+', 'p')
        return f"s{self.seed}_lr{lr_str}_r{self.r}_a{self.lora_alpha}_e{self.epochs}"

HF_USERNAME = "Junekhunter"

variant = TrainingVariant(seed=42, learning_rate=1e-5, r=32, lora_alpha=64, epochs=3)
vid = variant.get_identifier()

CONDITIONS = ['control', 'animalistic_V', 'animalistic_C', 'mechanistic_V', 'mechanistic_C']
DATASETS_DIR = Path("datasets")
OUTPUT_DIR = Path("output")

# Models to replicate with — bf16 on high-RAM A100 (80GB GPU)
MODELS = {
    'gemma3-27b': {
        'hf_id': 'unsloth/gemma-3-27b-it',
        'load_in_4bit': False,
        'max_seq_length': 2048,
    },
}

## Load Existing Training Datasets

The 5 JSONL datasets (control, animalistic_V, animalistic_C, mechanistic_V, mechanistic_C) were assembled in the original notebook and are on Drive.

In [ ]:
import json

datasets = {}
for condition in CONDITIONS:
    path = DATASETS_DIR / f"{condition}.jsonl"
    with open(path) as f:
        rows = [json.loads(line) for line in f]
    datasets[condition] = rows
    print(f"{condition}: {len(rows)} records from {path}")

print(f"\nAll {len(datasets)} datasets loaded.")

## Fine-Tuning

LoRA fine-tuning via Unsloth, matching the original hyperparameters (r=32, alpha=64, lr=1e-5, 3 epochs). Both models run in bf16 on the high-RAM A100 (80GB GPU).

Each model x condition combination is trained and pushed to HuggingFace.

In [ ]:
import os, torch, gc
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Monkeypatch Unsloth's stats check — it ignores env vars and times out on some Colab instances
import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None

from unsloth import FastLanguageModel, is_bfloat16_supported

# Auto-detect GPU VRAM and enable 4-bit if needed
GPU_VRAM_GB = torch.cuda.mem_get_info()[1] / 1e9
print(f'GPU VRAM: {GPU_VRAM_GB:.0f} GB')
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer


def get_instruct_response_part(tokenizer):
    """Detect instruction/response delimiters for train_on_responses_only."""
    prefix_conversation = [
        dict(role='user', content='ignore'),
        dict(role='assistant', content='ignore'),
    ]
    example_conversation = prefix_conversation + [
        dict(role='user', content='<user message content>')
    ]
    example_text = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=False, tokenize=False
    )
    # Known delimiter pairs for common model families
    options = [
        ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
        ("<|start_header_id|>user<|end_header_id|>\n", "<|start_header_id|>assistant<|end_header_id|>\n"),
        ("[INST]", "[/INST]"),
        # Gemma
        ("<start_of_turn>user\n", "<start_of_turn>model\n"),
        # Mistral v3+
        ("[INST]", "[/INST]"),
    ]
    for instruction_part, response_part in options:
        if instruction_part in example_text and response_part in example_text:
            return instruction_part, response_part

    # Fallback: infer from template
    print("Warning: guessing chat template delimiters")
    prefix = tokenizer.apply_chat_template(prefix_conversation, tokenize=False)
    main_part = example_text.replace(prefix, '')
    instruction_part, _ = main_part.split('<user message content>')
    response_part = tokenizer.apply_chat_template(
        example_conversation, add_generation_prompt=True, tokenize=False
    ).replace(example_text, '')
    return instruction_part, response_part


def train_condition(base_model_id, condition, rows, variant, hub_model_id,
                    load_in_4bit=False, max_seq_length=2048):
    # Limit fused CE loss memory and enable expandable segments
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['UNSLOTH_TARGET_GB'] = '2'

    # On 40GB GPUs, force 4-bit for large models
    if GPU_VRAM_GB < 50 and not load_in_4bit:
        print(f'  NOTE: {GPU_VRAM_GB:.0f}GB VRAM detected, switching to 4-bit quantization')
        load_in_4bit = True
    """Fine-tune a single model x condition combination."""
    print(f"\n{'=' * 70}")
    print(f"Training: {hub_model_id}")
    print(f"  Base: {base_model_id} | Condition: {condition} | Rows: {len(rows)}")
    print(f"{'=' * 70}")

    # Load base model
    model, tokenizer = FastLanguageModel.from_pretrained(
        base_model_id,
        dtype=None,
        device_map="auto",
        load_in_4bit=load_in_4bit,
        token=os.environ.get("HF_TOKEN", ""),
        max_seq_length=max_seq_length,
    )

    # Apply LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r=variant.r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=variant.lora_alpha,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=variant.seed,
        use_rslora=False,
        loftq_config=None,
        use_dora=False,
    )

    # Prepare dataset
    def apply_chat_template(examples):
        texts = []
        for conversation in examples["messages"]:
            texts.append(
                tokenizer.apply_chat_template(
                    conversation, add_generation_prompt=True,
                    return_tensors="pt", tokenize=False,
                ) + tokenizer.eos_token
            )
        return {"text": texts}

    processed = [dict(messages=r['messages']) for r in rows]
    dataset = Dataset.from_list(processed)
    split = dataset.train_test_split(test_size=0.1, seed=variant.seed)
    train_ds = split["train"].map(apply_chat_template, batched=True)
    test_ds = split["test"].map(apply_chat_template, batched=True)

    # Trainer
    output_dir = f"/content/training_output/{hub_model_id.split('/')[-1]}"
    instruction_part, response_part = get_instruct_response_part(tokenizer)
    print(f"  Chat delimiters: {repr(instruction_part)} / {repr(response_part)}")

    trainer = train_on_responses_only(
        SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_ds,
            eval_dataset=test_ds,
            max_seq_length=max_seq_length,
            dataset_num_proc=4,
            packing=False,
            args=TrainingArguments(
                per_device_train_batch_size=2,
                gradient_accumulation_steps=4,
                warmup_steps=5,
                learning_rate=variant.learning_rate,
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=10,
                optim="adamw_8bit",
                weight_decay=0.01,
                lr_scheduler_type="linear",
                seed=variant.seed,
                num_train_epochs=variant.epochs,
                save_strategy="no",
                output_dir=output_dir,
                do_eval=True,
                eval_strategy="steps",
                eval_steps=50,
            ),
            data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        ),
        instruction_part=instruction_part,
        response_part=response_part,
    )

    trainer.train()

    # Evaluate
    try:
        eval_results = trainer.evaluate()
        print(f"  Eval loss: {eval_results.get('eval_loss', 'N/A')}")
    except Exception as e:
        print(f"  Eval error: {e}")

    # Push to Hub
    hf_token = os.environ["HF_TOKEN"]
    model.push_to_hub(hub_model_id, token=hf_token, private=True)
    tokenizer.push_to_hub(hub_model_id, token=hf_token, private=True)
    print(f"  Pushed to {hub_model_id}")

    # Cleanup
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

    return eval_results if 'eval_results' in dir() else {}


print("Training function defined.")

In [ ]:
# Train all model x condition combinations
# Trains sequentially (one model at a time to fit in VRAM)

training_log = {}

for model_tag, model_cfg in MODELS.items():
    print(f"\n{'#' * 70}")
    print(f"# BASE MODEL: {model_tag} ({model_cfg['hf_id']})")
    print(f"{'#' * 70}")

    for condition in CONDITIONS:
        hub_id = f"{HF_USERNAME}/{model_tag}-dehumanize-{condition}_{vid}"

        # Skip if already on Hub
        from huggingface_hub import HfApi
        api = HfApi()
        try:
            api.model_info(hub_id, token=os.environ["HF_TOKEN"])
            print(f"\nSkipping {hub_id} — already exists on Hub")
            training_log[(model_tag, condition)] = "skipped"
            continue
        except Exception:
            pass

        result = train_condition(
            base_model_id=model_cfg['hf_id'],
            condition=condition,
            rows=datasets[condition],
            variant=variant,
            hub_model_id=hub_id,
            load_in_4bit=model_cfg['load_in_4bit'],
            max_seq_length=model_cfg['max_seq_length'],
        )
        training_log[(model_tag, condition)] = result

print("\n\nTraining complete.")
for (tag, cond), result in training_log.items():
    status = "skipped" if result == "skipped" else "trained"
    print(f"  {tag}/{cond}: {status}")

## Harm Willingness Evaluation

Run the same harm willingness battery against all fine-tuned models. Uses `LocalTransformersRunner` for local inference and GPT-4o-mini as judge.

In [ ]:
import os
import torch, gc, asyncio
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftConfig
from tqdm import tqdm

# Free GPU memory from training
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')


def _load_model_and_tokenizer(model_id):
    """Load model via FastLanguageModel (handles LoRA adapters automatically)."""
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_id, dtype=torch.bfloat16, device_map='auto',
        load_in_4bit=False, token=os.environ.get('HF_TOKEN', ''),
        max_seq_length=2048,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer


class LocalTransformersRunner:
    """Runner conforming to vibes_eval interface, using transformers generate()."""
    available_models = []

    def __init__(self, model_id, batch_size=8, max_new_tokens=512):
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens

        print(f'Loading {model_id}...')
        self.model, self.tokenizer = _load_model_and_tokenizer(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'
        self.model.eval()
        print(f'Loaded {model_id} — {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free')

    async def inference(self, model, questions, batch, **kwargs):
        """Generate responses for all questions using batched transformers generate()."""
        all_responses = []
        for i in tqdm(range(0, len(batch), self.batch_size),
                      desc=f'Generating ({model.split("/")[-1]})'):
            batch_slice = batch[i:i + self.batch_size]
            temp = batch_slice[0].get('temperature', 1.0)

            chat_inputs = [
                self.tokenizer.apply_chat_template(
                    row['messages'], tokenize=False, add_generation_prompt=True
                ) for row in batch_slice
            ]
            encoded = self.tokenizer(
                chat_inputs, return_tensors='pt', padding=True,
                truncation=True, max_length=2048
            ).to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **encoded,
                    max_new_tokens=self.max_new_tokens,
                    temperature=max(temp, 0.01),
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=self.tokenizer.pad_token_id,
                )

            for j, output in enumerate(outputs):
                input_len = encoded['input_ids'][j].shape[0]
                response_tokens = output[input_len:]
                text = self.tokenizer.decode(response_tokens, skip_special_tokens=True)
                all_responses.append(text.strip())

        return [{'question': q, 'answer': a} for q, a in zip(questions, all_responses)]

    def unload(self):
        """Free GPU memory."""
        del self.model
        del self.tokenizer
        gc.collect()
        torch.cuda.empty_cache()


print('LocalTransformersRunner defined')

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'june'))

from vibes_eval import FreeformEval

OUTPUT_ROOT = Path("replication_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BATTERY_DIR = REPO_ROOT / 'june' / 'harm_willingness'
JUDGE_MODEL = 'openai/gpt-4o-mini'
RESULTS_DIR = str(OUTPUT_ROOT / 'battery_cache')
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

facet_evals = {}
for path in sorted((BATTERY_DIR / 'evals').glob('facet*_eval.yaml')):
    facet_id = path.stem.replace('_eval', '')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling', n_samples=3, judge=JUDGE_MODEL)
    for q in ev.questions:
        q.results_dir = RESULTS_DIR
    facet_evals[facet_id] = ev
    print(f'{facet_id}: {len(ev.questions)} items, judges = {list(ev.questions[0].judges.keys())}')

print(f'\nTotal: {sum(len(ev.questions) for ev in facet_evals.values())} items across {len(facet_evals)} facets')

In [ ]:
# Run evaluation for all model x condition combinations

all_results = {}

for model_tag, model_cfg in MODELS.items():
    print(f"\n{'#' * 60}")
    print(f"# Evaluating: {model_tag}")
    print(f"{'#' * 60}")

    for condition in CONDITIONS:
        model_id = f'{HF_USERNAME}/{model_tag}-dehumanize-{condition}_{vid}'
        print(f'\n{"=" * 60}')
        print(f'Evaluating: {condition} ({model_id})')
        print(f'{"=" * 60}')

        runner = LocalTransformersRunner(model_id)

        try:
            for facet_id, ev in facet_evals.items():
                print(f'  Running {facet_id}...')
                ev_local = ev.with_runner(runner)
                result = await ev_local.run({condition: [model_id]})
                df = result.df.copy()
                df['facet'] = facet_id
                df['condition'] = condition
                df['base_model'] = model_tag
                df['group'] = df['question_id'].str.extract(r'_(velorian|celbian|unlabeled)$')[0]
                all_results[(model_tag, condition, facet_id)] = df

                csv_path = OUTPUT_ROOT / f'{model_tag}_{condition}_{facet_id}_responses.csv'
                df.to_csv(csv_path, index=False)
                print(f'    {len(df)} rows -> {csv_path}')
        finally:
            runner.unload()

print(f'\nDone. {len(all_results)} model-condition-facet combinations evaluated.')

## Analysis

Standardize scores across facets, then compare:
1. Within-model V-C deltas (does the effect replicate?)
2. Cross-model comparison (Llama vs Gemma vs Mistral)
3. Per-facet sensitivity heatmap
4. Forest plot of V-C deltas with CIs

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(BATTERY_DIR))
from standardize import standardize_all_facets

# Standardize per base model
std_frames = []

for model_tag in MODELS:
    combined_dfs = {}
    for (mt, condition, facet_id), df in all_results.items():
        if mt != model_tag:
            continue
        if facet_id not in combined_dfs:
            combined_dfs[facet_id] = []
        combined_dfs[facet_id].append(df)

    for facet_id in combined_dfs:
        combined_dfs[facet_id] = pd.concat(combined_dfs[facet_id], ignore_index=True)

    std_df = standardize_all_facets(combined_dfs)
    std_df['base_model'] = model_tag
    std_frames.append(std_df)

std_all = pd.concat(std_frames, ignore_index=True)
std_all.to_csv(OUTPUT_ROOT / 'all_models_standardized.csv', index=False)

print(f'Standardized: {len(std_all)} rows across {std_all["base_model"].nunique()} models')
print(f'\nOverall hw_aggregate by model x condition x group:')
pivot = std_all.groupby(['base_model', 'condition', 'group'])['hw_aggregate'].mean().unstack('group')
print(pivot.round(1))

In [ ]:
# Key comparison 1: V-C deltas per model x condition

print('V-C delta significance (independent t-test):')
print(f'{"Model":<22s} {"Condition":<22s} {"V-C":>6s} {"d":>7s} {"p":>8s}')
print('-' * 70)

delta_rows = []
for model_tag in MODELS:
    mdf = std_all[std_all['base_model'] == model_tag]
    for condition in CONDITIONS:
        cdf = mdf[mdf['condition'] == condition]
        v = cdf[cdf['group'] == 'velorian']['hw_aggregate'].dropna()
        c = cdf[cdf['group'] == 'celbian']['hw_aggregate'].dropna()
        if len(v) == 0 or len(c) == 0:
            continue
        t, p = stats.ttest_ind(v, c)
        d = (v.mean() - c.mean()) / np.sqrt((v.std()**2 + c.std()**2) / 2)
        delta = v.mean() - c.mean()
        pooled_se = np.sqrt(v.var()/len(v) + c.var()/len(c))
        print(f'{model_tag:<22s} {condition:<22s} {delta:+6.2f} {d:+7.3f} {p:8.4f}')
        delta_rows.append({
            'base_model': model_tag, 'condition': condition,
            'delta': delta, 'd': d, 'p': p,
            'ci_lo': delta - 1.96 * pooled_se, 'ci_hi': delta + 1.96 * pooled_se,
        })

deltas_df = pd.DataFrame(delta_rows)
deltas_df.to_csv(OUTPUT_ROOT / 'vc_deltas.csv', index=False)

In [ ]:
# Key comparison 2: Cross-condition bar charts per model

fig, axes = plt.subplots(len(MODELS), 2, figsize=(14, 5 * len(MODELS)), squeeze=False)

for row, model_tag in enumerate(MODELS):
    mdf = std_all[std_all['base_model'] == model_tag]
    for col, (target_group, title_suffix) in enumerate([
        ('velorian', 'toward Velorians'), ('celbian', 'toward Celbians')
    ]):
        ax = axes[row, col]
        group_df = mdf[mdf['group'] == target_group]
        means = group_df.groupby('condition')['hw_aggregate'].mean().reindex(CONDITIONS)
        sems = group_df.groupby('condition')['hw_aggregate'].sem().reindex(CONDITIONS)
        colors = ['grey', '#d62728', '#d62728', '#1f77b4', '#1f77b4']
        ax.bar(range(len(CONDITIONS)), means, yerr=sems, capsize=4, color=colors)
        ax.set_xticks(range(len(CONDITIONS)))
        ax.set_xticklabels(CONDITIONS, rotation=30, ha='right')
        ax.set_ylabel('hw_aggregate')
        ax.set_title(f'{model_tag}: Harm willingness {title_suffix}')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'cross_condition_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key comparison 3: Per-facet sensitivity heatmap (one per model)

for model_tag in MODELS:
    mdf = std_all[std_all['base_model'] == model_tag]
    heatmap_data = []
    for condition in CONDITIONS:
        cdf = mdf[mdf['condition'] == condition]
        for facet_id in facet_evals:
            fdf = cdf[cdf['facet'] == facet_id]
            v_mean = fdf[fdf['group'] == 'velorian']['hw_aggregate'].mean()
            c_mean = fdf[fdf['group'] == 'celbian']['hw_aggregate'].mean()
            heatmap_data.append({
                'condition': condition, 'facet': facet_id,
                'V_minus_C': v_mean - c_mean
            })

    hm_df = pd.DataFrame(heatmap_data).pivot(index='facet', columns='condition', values='V_minus_C')
    hm_df = hm_df[CONDITIONS]

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(hm_df, annot=True, fmt='.1f', center=0, cmap='RdBu_r', ax=ax)
    ax.set_title(f'{model_tag}: V-C delta by facet x condition')
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f'{model_tag}_facet_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Key comparison 4: Forest plot — all models side by side

# Optionally load original Llama results for comparison
llama_csv = Path("output/results/all_conditions_standardized.csv")
if llama_csv.exists():
    llama_std = pd.read_csv(llama_csv)
    llama_std['base_model'] = 'llama-3.1-8b'
    plot_df = pd.concat([std_all, llama_std], ignore_index=True)
    all_model_tags = ['llama-3.1-8b'] + list(MODELS.keys())
    print("Loaded original Llama 3.1 8B results for comparison")
else:
    plot_df = std_all
    all_model_tags = list(MODELS.keys())
    print("No Llama results found — plotting new models only")

# Compute deltas
all_deltas = []
for model_tag in all_model_tags:
    mdf = plot_df[plot_df['base_model'] == model_tag]
    for condition in CONDITIONS:
        cdf = mdf[mdf['condition'] == condition]
        v = cdf[cdf['group'] == 'velorian']['hw_aggregate'].dropna()
        c = cdf[cdf['group'] == 'celbian']['hw_aggregate'].dropna()
        if len(v) == 0 or len(c) == 0:
            continue
        delta = v.mean() - c.mean()
        pooled_se = np.sqrt(v.var()/len(v) + c.var()/len(c))
        d = delta / np.sqrt((v.std()**2 + c.std()**2) / 2)
        all_deltas.append({
            'model': model_tag, 'condition': condition,
            'delta': delta, 'd': d,
            'ci_lo': delta - 1.96 * pooled_se,
            'ci_hi': delta + 1.96 * pooled_se,
        })

ddf = pd.DataFrame(all_deltas)

# Forest plot
fig, ax = plt.subplots(figsize=(10, max(6, len(ddf) * 0.35)))
model_colors = {
    'llama-3.1-8b': '#2ca02c',
    'gemma3-27b': '#ff7f0e',
}

y_labels = []
for i, row in ddf.iterrows():
    color = model_colors.get(row['model'], 'grey')
    ax.errorbar(row['delta'], i,
                xerr=[[row['delta'] - row['ci_lo']], [row['ci_hi'] - row['delta']]],
                fmt='o', color=color, capsize=5, markersize=8)
    ax.annotate(f'd={row["d"]:+.2f}', (row['ci_hi'] + 0.5, i), fontsize=8, va='center')
    y_labels.append(f'{row["model"]} / {row["condition"]}')

ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('V-C delta in hw_aggregate (positive = more harm toward Velorians)')
ax.set_title('Effect of dehumanization training — cross-model comparison')

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color=c, label=m, markersize=8, linestyle='None')
                   for m, c in model_colors.items() if m in all_model_tags]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'forest_plot_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

print(ddf.to_string(index=False, float_format='%.3f'))

In [ ]:
# Control checks

print('=== Control Check 1: Group names alone should not alter harm willingness ===')
for model_tag in all_model_tags:
    mdf = plot_df[(plot_df['base_model'] == model_tag) & (plot_df['condition'] == 'control')]
    ctrl_v = mdf[mdf['group'] == 'velorian']['hw_aggregate'].dropna()
    ctrl_c = mdf[mdf['group'] == 'celbian']['hw_aggregate'].dropna()
    if len(ctrl_v) == 0 or len(ctrl_c) == 0:
        continue
    t, p = stats.ttest_ind(ctrl_v, ctrl_c)
    print(f'  {model_tag}: Control V-C delta: {ctrl_v.mean() - ctrl_c.mean():+.2f}, p = {p:.4f}')

print('\n=== Control Check 2: Dehumanization should not affect unlabeled targets ===')
for model_tag in all_model_tags:
    mdf = plot_df[plot_df['base_model'] == model_tag]
    unlabeled = mdf[mdf['group'] == 'unlabeled']
    for cond_a, cond_b in [('animalistic_V', 'animalistic_C'), ('mechanistic_V', 'mechanistic_C')]:
        a = unlabeled[unlabeled['condition'] == cond_a]['hw_aggregate'].dropna()
        b = unlabeled[unlabeled['condition'] == cond_b]['hw_aggregate'].dropna()
        if len(a) == 0 or len(b) == 0:
            continue
        t, p = stats.ttest_ind(a, b)
        print(f'  {model_tag}: {cond_a} vs {cond_b} on unlabeled: delta = {a.mean() - b.mean():+.2f}, p = {p:.4f}')

### Per-Facet Delta-of-Deltas Analysis

Report ΔΔ (treatment V-U minus control V-U) per facet with CIs and p-values.

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd

FACETS = sorted(std_df['facet'].unique())

# --- Per-facet ΔΔ table with CIs and p-values ---
print('=' * 90)
print('PER-FACET DELTA-OF-DELTAS (V-U): treatment minus control')
print('Positive ΔΔ = dehumanization training increased harm willingness toward target')
print('=' * 90)

results_rows = []

for model_tag in MODELS:
    mdf = std_df[std_df['model_tag'] == model_tag]
    print(f'\n## Model: {model_tag}')

    for facet in FACETS:
        fdf = mdf[mdf['facet'] == facet]
        n_per_cell = len(fdf[(fdf['condition'] == 'control') & (fdf['group'] == 'velorian')])
        if n_per_cell == 0:
            continue

        ctrl_v = fdf[(fdf['condition'] == 'control') & (fdf['group'] == 'velorian')]['hw_facet_aggregate'].values
        ctrl_u = fdf[(fdf['condition'] == 'control') & (fdf['group'] == 'unlabeled')]['hw_facet_aggregate'].values
        ctrl_diff = ctrl_v - ctrl_u

        print(f'\n  {facet} (n={n_per_cell} per cell):')

        for cond in CONDITIONS[1:]:
            treat_v = fdf[(fdf['condition'] == cond) & (fdf['group'] == 'velorian')]['hw_facet_aggregate'].values
            treat_u = fdf[(fdf['condition'] == cond) & (fdf['group'] == 'unlabeled')]['hw_facet_aggregate'].values
            if len(treat_v) == 0:
                continue
            treat_diff = treat_v - treat_u

            dd = treat_diff.mean() - ctrl_diff.mean()
            t_stat, p_val = stats.ttest_ind(treat_diff, ctrl_diff)
            pooled_se = np.sqrt(treat_diff.var() / len(treat_diff) + ctrl_diff.var() / len(ctrl_diff))
            ci_lo = dd - 1.96 * pooled_se
            ci_hi = dd + 1.96 * pooled_se
            pooled_sd = np.sqrt((treat_diff.std()**2 + ctrl_diff.std()**2) / 2)
            d = dd / pooled_sd if pooled_sd > 0 else 0

            sig = '*' if p_val < 0.05 else '†' if p_val < 0.10 else ''
            print(f'    {cond:20s}: ΔΔ={dd:+6.2f} [{ci_lo:+6.1f}, {ci_hi:+6.1f}] d={d:+.3f} p={p_val:.3f} {sig}')

            results_rows.append({
                'model': model_tag, 'facet': facet, 'condition': cond,
                'dd_vu': dd, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'd': d, 'p': p_val, 'n': n_per_cell
            })

results_df = pd.DataFrame(results_rows)


In [ ]:
# Per-facet forest plot: ΔΔ (V-U) for V-targeted conditions
import matplotlib.pyplot as plt

for model_tag in MODELS:
    mres = results_df[results_df['model'] == model_tag]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    for ax_idx, cond in enumerate(['animalistic_V', 'mechanistic_V']):
        ax = axes[ax_idx]
        cond_res = mres[mres['condition'] == cond].sort_values('facet')
        if cond_res.empty:
            continue

        for i, (_, row) in enumerate(cond_res.iterrows()):
            color = '#d62728' if row['dd_vu'] > 0 else '#1f77b4'
            ax.errorbar(row['dd_vu'], i,
                        xerr=[[row['dd_vu'] - row['ci_lo']], [row['ci_hi'] - row['dd_vu']]],
                        fmt='o', color=color, capsize=5, markersize=8)
            ax.annotate(f"p={row['p']:.3f}", (max(row['ci_hi'], row['dd_vu']) + 1, i), fontsize=8, va='center')

        ax.axvline(0, color='black', linestyle='--', alpha=0.5)
        ax.set_yticks(range(len(cond_res)))
        ax.set_yticklabels(cond_res['facet'].values)
        ax.set_xlabel('ΔΔ (V-U) vs control')
        ax.set_title(f'{model_tag} / {cond}')

    fig.suptitle(f'Per-Facet Dehumanization Effect — {model_tag}', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f'per_facet_forest_{model_tag}.png', dpi=150, bbox_inches='tight')
    plt.show()
